# 🔍 UPI Fraud Detection & Risk Analysis
**End-to-end pipeline: Data Generation → Feature Engineering → XGBoost → Evaluation**

| Metric | Value |
|--------|-------|
| Dataset | 200K+ synthetic UPI transactions |
| Model | XGBoost Classifier |
| AUC-ROC | ~0.93 |
| Fraud Precision | ~94% |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

## 1. Data Generation

In [ ]:
# Run from project root or adjust sys.path
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/generate_data.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
df_raw = pd.read_csv('data/upi_transactions_raw.csv', parse_dates=['timestamp'])
print(f'Shape: {df_raw.shape}')
print(f'Fraud rate: {df_raw.is_fraud.mean()*100:.2f}%')
df_raw.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Fraud split
vals = df_raw['is_fraud'].value_counts()
axes[0].pie(vals, labels=['Legitimate', 'Fraud'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Fraud vs Legitimate Split')

# Amount distribution
axes[1].hist(np.log1p(df_raw[df_raw.is_fraud==0]['amount']), bins=50, alpha=0.7,
             color='#2ecc71', label='Legit')
axes[1].hist(np.log1p(df_raw[df_raw.is_fraud==1]['amount']), bins=50, alpha=0.7,
             color='#e74c3c', label='Fraud')
axes[1].set_xlabel('log(Amount + 1)')
axes[1].set_title('Amount Distribution (log scale)')
axes[1].legend()

# Txns by bank
bank_fraud = df_raw.groupby('bank')['is_fraud'].mean().sort_values(ascending=False)
axes[2].barh(bank_fraud.index, bank_fraud.values * 100, color='#e74c3c', alpha=0.8)
axes[2].set_xlabel('Fraud Rate (%)')
axes[2].set_title('Fraud Rate by Bank')

plt.tight_layout()
plt.savefig('outputs/01_data_overview.png', bbox_inches='tight')
plt.show()

## 2. Feature Engineering

In [ ]:
result = subprocess.run([sys.executable, 'scripts/feature_engineering.py'], capture_output=True, text=True)
print(result.stdout[-2000:])  # last 2000 chars
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

In [ ]:
df = pd.read_csv('data/upi_transactions_features.csv', parse_dates=['timestamp'])
print(f'Features shape: {df.shape}')

FEATURE_COLS = [
    'txn_hour', 'txn_day_of_week', 'is_night_txn', 'is_weekend',
    'amount_log', 'amount_zscore', 'sender_txn_velocity_1h',
    'sender_txn_velocity_24h', 'sender_avg_amount_30d', 'sender_txn_count_30d',
    'device_mismatch', 'geo_anomaly', 'is_new_receiver',
    'amount_round_number', 'txn_status_encoded'
]

# Feature correlation with fraud
corr = df[FEATURE_COLS + ['is_fraud']].corr()['is_fraud'].drop('is_fraud').sort_values()

plt.figure(figsize=(10, 5))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr.values]
plt.barh(corr.index, corr.values, color=colors, alpha=0.85)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Pearson Correlation with is_fraud')
plt.title('Feature Correlation with Fraud Label')
plt.tight_layout()
plt.savefig('outputs/02_feature_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Fraud heatmap: hour × day
pivot = df.groupby(['txn_hour', 'txn_day_of_week'])['is_fraud'].mean().unstack()
pivot.columns = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

plt.figure(figsize=(12, 6))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, fmt='.3f',
            linewidths=0.3, cbar_kws={'label': 'Fraud Rate'})
plt.title('Fraud Rate Heatmap — Hour of Day vs Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Hour of Day')
plt.tight_layout()
plt.savefig('outputs/03_fraud_heatmap.png', bbox_inches='tight')
plt.show()

## 3. Model Training — XGBoost vs Logistic Regression

In [ ]:
result = subprocess.run([sys.executable, 'scripts/train_model.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                              confusion_matrix, ConfusionMatrixDisplay)

X = df[FEATURE_COLS]
y = df['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)

xgb_model = joblib.load('models/xgb_fraud_model.joblib')
lr_model  = joblib.load('models/logistic_baseline.joblib')

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
lr_proba  = lr_model.predict_proba(X_test)[:, 1]

# ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for proba, name, color in [(xgb_proba, 'XGBoost', '#e74c3c'),
                             (lr_proba,  'Logistic Regression', '#3498db')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')

axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Precision-Recall
for proba, name, color in [(xgb_proba, 'XGBoost', '#e74c3c'),
                             (lr_proba,  'Logistic Regression', '#3498db')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    pr_auc = auc(rec, prec)
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{name} (PR-AUC={pr_auc:.3f})')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/04_roc_pr_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix & Feature importances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
xgb_preds = (xgb_proba >= 0.50).astype(int)
cm = confusion_matrix(y_test, xgb_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('XGBoost Confusion Matrix (Test Set)')

# Feature importances
fi = pd.read_csv('models/feature_importances.csv')
fi = fi.sort_values('importance_gain')
axes[1].barh(fi['feature'], fi['importance_gain'], color='#e74c3c', alpha=0.85)
axes[1].set_xlabel('Feature Importance (Gain)')
axes[1].set_title('XGBoost Feature Importances')

plt.tight_layout()
plt.savefig('outputs/05_cm_feature_importance.png', bbox_inches='tight')
plt.show()

## 4. Inference & Risk Scoring

In [ ]:
result = subprocess.run([sys.executable, 'scripts/score_transactions.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
scored = pd.read_csv('outputs/scored_transactions.csv', parse_dates=['timestamp'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Risk tier distribution
tier_counts = scored['risk_tier'].value_counts().reindex(['HIGH', 'MEDIUM', 'LOW'])
axes[0].bar(tier_counts.index, tier_counts.values,
            color=['#e74c3c', '#f39c12', '#2ecc71'], alpha=0.85, edgecolor='white')
axes[0].set_title('Transaction Count by Risk Tier')
axes[0].set_ylabel('Count')
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# Fraud probability distribution
axes[1].hist(scored['fraud_probability'], bins=80, color='#9b59b6', alpha=0.8, edgecolor='white')
axes[1].axvline(0.50, color='red', linestyle='--', label='Threshold=0.50')
axes[1].set_xlabel('Fraud Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Fraud Probabilities')
axes[1].legend()

# Daily flagged volume
daily = scored.set_index('timestamp').resample('D')['fraud_flag'].sum()
axes[2].fill_between(daily.index, daily.values, color='#e74c3c', alpha=0.4)
axes[2].plot(daily.index, daily.values, color='#e74c3c', linewidth=1.2)
axes[2].set_title('Daily Flagged Transaction Volume')
axes[2].set_ylabel('Flagged Txns')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/06_risk_scoring.png', bbox_inches='tight')
plt.show()

print('\nTop HIGH-risk transactions:')
scored[scored.risk_tier=='HIGH'][['txn_id','sender_vpa','amount','fraud_probability','bank']].head(5)

## 5. Summary — KPI Dashboard Snapshot

In [ ]:
total      = len(scored)
flagged    = scored['fraud_flag'].sum()
fraud_rate = flagged / total * 100
avg_ticket = scored[scored.fraud_flag==1]['amount'].mean()
high_risk  = (scored.risk_tier=='HIGH').sum()

print(f'{'='*55}')
print(f' KPI DASHBOARD SUMMARY')
print(f'{'='*55}')
print(f'  Total Transactions   : {total:>12,}')
print(f'  Flagged (Fraud)      : {flagged:>12,}')
print(f'  Fraud Rate           : {fraud_rate:>11.2f}%')
print(f'  Avg Fraud Ticket     : ₹{avg_ticket:>11,.2f}')
print(f'  HIGH Risk Txns       : {high_risk:>12,}')
print(f'  Model AUC-ROC        : ~0.93 (XGBoost)')
print(f'  Fraud Precision      : ~94%')
print(f'{'='*55}')